In [11]:
import os
import pandas as pd
import numpy as np
import glob

In [5]:
save_data_flag = False

## Bead prediction task

In [7]:
beads_datadir = "./Data/beads/"
beads_rawdatadir = beads_datadir + "raw/"

datafiles = [f for f in os.listdir(beads_rawdatadir) if os.path.isfile(beads_rawdatadir + f)]

# make a csv with trial-wise data for each subject, to be used for pre-processing and analyses
preprocdf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'run': [],
    'trial': [],
    'jar': [],
    'bead': [],
    'choice': []
})

for ii,sub in enumerate(datafiles):

    df = pd.read_csv(beads_rawdatadir + sub)

    # necessary to ensure same ordering of sj index
    sjid = df['subject_id'].iloc[0]
    sjind = ii
    # sjind = ibdf[ibdf['subject_ID']==sjid]['subject_index'].iloc[0] # REMOVE

    datadf = df[df['block'] == 'lowH']

    for run in [1,2]:
        choices = datadf.loc[datadf['block_rep'] == run, 'choice'].to_numpy()
        jars = datadf.loc[datadf['block_rep'] == run, 'jar'].to_numpy()
        beads = datadf.loc[datadf['block_rep'] == run, 'bead'].to_numpy()
        trials = datadf.loc[datadf['block_rep'] == run, 'trial_number'].to_numpy()

        sjdf = pd.DataFrame({
            'subject_ID': [sjid] * len(trials),
            'subject_index': [sjind] * len(trials),
            'run': [run] * len(trials),
            'trial': trials,
            'jar': jars,
            'bead': beads,
            'choice': choices
        })

        preprocdf = pd.concat((preprocdf,sjdf),ignore_index=True)

if save_data_flag:
    preprocdf.to_csv(beads_datadir + 'sj-preproc-data.csv',index=False)

## Horse prediction experiments

In [12]:
horses_datadir = "./Data/horses/"

### Low WS ratio experiment

In [13]:
horses_datadir_low = horses_datadir + "lowWS/"

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'exclude_Ixr': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed',
    'key_resp_test.corr',
    'key_resp_6.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

renamed_columns = [
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

dir_list = [
    horses_datadir_low + 'raw/*.csv',
    horses_datadir_low + 'raw/exclude/*.csv',
]

sjindex = 0
dirindex = 0

# get non-excluded subjects
for dir_ in dir_list:
    for fname in glob.glob(dir_):

        sjdf = pd.DataFrame({})

        df = pd.read_csv(fname)
        sjid = df['PROLIFIC_PID'].iloc[0]
        dist2_side = df['dist2_loc'].iloc[0]
        # if df['dist1_loc'].iloc[0] == 'left':
        #     choice_map = dict({'left': 1, 'right': 2})
        # else:
        #     choice_map = dict({'right': 1, 'left': 2})

        # run 1
        datadf = df[df['key_resp_test.keys'].notna()]
        datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
        datadf = datadf[run1_columns]
        datadf.rename(columns=run1_map,inplace=True)
        sjdf = pd.concat((sjdf,datadf),ignore_index=True)

        sjdf['choice'] = sjdf['choice'].astype(float) + 1
        sjdf['subject_ID'] = sjid
        sjdf['subject_index'] = sjindex
        sjdf['trial_index'] = sjdf['trial_index'] - 100
        if dirindex > 0:
            sjdf['exclude_Ixr'] = 1
        else:
            sjdf['exclude_Ixr'] = 0

        sjindex += 1

        fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

    dirindex += 1

fulldf.replace(shape_map_all,inplace=True)

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

if save_data_flag:
    fulldf.to_csv(horses_datadir_low + 'sj-preproc-data.csv',index=False)

/tmp/ipykernel_38932/524187704.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
/tmp/ipykernel_38932/524187704.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
/tmp/ipykernel_38932/524187704.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in th

### Intermediate WS ratio / learning experiment

In [16]:
horses_datadir_mid = horses_datadir + "midWS_learning/"

fulldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'age': [],
    'exclude_Ixr': [],
    'condition': [],
    'trial_ID': [],
    'trial_index': [],
    'observation_1': [],
    'observation_2': [],
    'observation_3': [],
    'observation_4': [],
    'observation_5': [],
    'latent_state': [],
    'optimal_choice': [],
    'correct': [],
    'intertrial_duration': [],
    'trial_start_time': [],
    'choice': [],
    'rt': []
})

run1_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed_1',
    'key_resp_test.corr',
    'key_proceed_1.rt',
    'testing_trial.started',
    'choice', # added in the loop below
    'key_resp_test.rt'
]

run2_columns = [
    'trial_index',
    'shape1',
    'shape2',
    'shape3',
    'shape4',
    'shape5',
    'distr',
    'pred_distr',
    'num_completed_2',
    'key_resp_test_5.corr',
    'key_proceed_2.rt',
    'testing_trial_2.started',
    'choice',
    'key_resp_test_5.rt'
]

renamed_columns = [
    'trial_ID',
    'observation_1',
    'observation_2',
    'observation_3',
    'observation_4',
    'observation_5',
    'latent_state',
    'optimal_choice',
    'trial_index',
    'correct',
    'intertrial_duration',
    'trial_start_time',
    'choice',
    'rt'
]

run1_map = dict(zip(run1_columns,renamed_columns))
run2_map = dict(zip(run2_columns,renamed_columns))

shape_map = {
    'shape_1': 1.0,
    'shape_2': 2.0,
    'shape_3': 3.0,
    'shape_4': 4.0
}

shape_map_all = {
    'observation_1': shape_map,
    'observation_2': shape_map,
    'observation_3': shape_map,
    'observation_4': shape_map,
    'observation_5': shape_map
}

sjindex = 0

# get non-excluded subjects
for fname in glob.glob(horses_datadir_mid + 'raw/*.csv'):

    sjdf = pd.DataFrame({})

    df = pd.read_csv(fname)
    sjid = df['PROLIFIC_PID'].iloc[0]
    dist2_side = df['dist2_loc'].iloc[0]
    age = df['info_survey.question6'].iloc[0]
    # if df['dist1_loc'].iloc[0] == 'left':
    #     choice_map = dict({'left': 1, 'right': 2})
    # else:
    #     choice_map = dict({'right': 1, 'left': 2})

    # run 1
    datadf = df[df['key_resp_test.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
    datadf = datadf[run1_columns]
    datadf.rename(columns=run1_map,inplace=True)
    datadf['condition'] = 'run_1'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    # run 2
    datadf = df[df['key_resp_test_5.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
    datadf = datadf[run2_columns]
    datadf.rename(columns=run2_map,inplace=True)
    datadf['condition'] = 'run_2'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    sjdf['choice'] = sjdf['choice'].astype(float) + 1
    sjdf['subject_ID'] = sjid
    sjdf['subject_index'] = sjindex
    sjdf['age'] = age
    sjdf['exclude_Ixr'] = 0

    sjindex += 1

    fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

# get subjects excluded for information capacity < 0.05 for both conditions
for fname in glob.glob(horses_datadir_mid + 'raw/exclude/*.csv'):
    
    sjdf = pd.DataFrame({})

    df = pd.read_csv(fname)
    sjid = df['PROLIFIC_PID'].iloc[0]
    dist2_side = df['dist2_loc'].iloc[0]
    age = df['info_survey.question6'].iloc[0]
    # if df['dist1_loc'].iloc[0] == 'left':
    #     choice_map = dict({'left': 1, 'right': 2})
    # else:
    #     choice_map = dict({'right': 1, 'left': 2})

    # run 1
    datadf = df[df['key_resp_test.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
    datadf = datadf[run1_columns]
    datadf.rename(columns=run1_map,inplace=True)
    datadf['condition'] = 'run_1'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    # run 2
    datadf = df[df['key_resp_test_5.keys'].notna()]
    datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
    datadf = datadf[run2_columns]
    datadf.rename(columns=run2_map,inplace=True)
    datadf['condition'] = 'run_2'
    sjdf = pd.concat((sjdf,datadf),ignore_index=True)

    sjdf['choice'] = sjdf['choice'].astype(float) + 1
    sjdf['subject_ID'] = sjid
    sjdf['subject_index'] = sjindex
    sjdf['age'] = age
    sjdf['exclude_Ixr'] = 1

    sjindex += 1

    fulldf = pd.concat((fulldf,sjdf),ignore_index=True)

fulldf.replace(shape_map_all,inplace=True)

fulldf['observation_encoding'] = \
    10**(fulldf['observation_1']-1) + \
    10**(fulldf['observation_2']-1) + \
    10**(fulldf['observation_3']-1) + \
    10**(fulldf['observation_4']-1) + \
    10**(fulldf['observation_5']-1)

fulldf['observation_encoding_ew'] = \
    10**(fulldf['observation_1']>2) + \
    10**(fulldf['observation_2']>2) + \
    10**(fulldf['observation_3']>2) + \
    10**(fulldf['observation_4']>2) + \
    10**(fulldf['observation_5']>2)

no_strong_flag = \
    ~( \
    (fulldf['observation_1']==1) | (fulldf['observation_1']==4) | \
    (fulldf['observation_2']==1) | (fulldf['observation_2']==4) | \
    (fulldf['observation_3']==1) | (fulldf['observation_3']==4) | \
    (fulldf['observation_4']==1) | (fulldf['observation_4']==4) | \
    (fulldf['observation_5']==1) | (fulldf['observation_5']==4) \
        )

fulldf['observation_encoding_iw'] = \
    (10**(fulldf['observation_1']-1)) * ((fulldf['observation_1']==1) | (fulldf['observation_1']==4)+no_strong_flag) + \
    (10**(fulldf['observation_2']-1)) * ((fulldf['observation_2']==1) | (fulldf['observation_2']==4)+no_strong_flag) + \
    (10**(fulldf['observation_3']-1)) * ((fulldf['observation_3']==1) | (fulldf['observation_3']==4)+no_strong_flag) + \
    (10**(fulldf['observation_4']-1)) * ((fulldf['observation_4']==1) | (fulldf['observation_4']==4)+no_strong_flag) + \
    (10**(fulldf['observation_5']-1)) * ((fulldf['observation_5']==1) | (fulldf['observation_5']==4)+no_strong_flag)

fulldf

/tmp/ipykernel_38932/2938866103.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test.keys'] == dist2_side
/tmp/ipykernel_38932/2938866103.py:120: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datadf['choice'] = datadf.loc[:,'key_resp_test_5.keys'] == dist2_side
/tmp/ipykernel_38932/2938866103.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cavea

,subject_ID,subject_index,age,exclude_Ixr,condition,trial_ID,trial_index,observation_1,observation_2,observation_3,...,latent_state,optimal_choice,correct,intertrial_duration,trial_start_time,choice,rt,observation_encoding,observation_encoding_ew,observation_encoding_iw
0,673d6028f83d1741dc6af737,0.0,23.0,0.0,run_1,276.0,0.0,2.0,3.0,2.0,...,2.0,2.0,1.0,0.2872,691.1780,2.0,0.2502,230.0,23,230.0
1,673d6028f83d1741dc6af737,0.0,23.0,0.0,run_1,263.0,1.0,2.0,1.0,3.0,...,2.0,1.0,0.0,0.2459,693.4432,1.0,0.2558,2111.0,32,2001.0
2,673d6028f83d1741dc6af737,0.0,23.0,0.0,run_1,285.0,2.0,2.0,2.0,3.0,...,2.0,1.0,1.0,0.5769,695.6743,2.0,0.2078,1130.0,23,1000.0
3,673d6028f83d1741dc6af737,0.0,23.0,0.0,run_1,199.0,3.0,1.0,4.0,3.0,...,2.0,2.0,1.0,0.2957,698.1726,2.0,0.2596,1112.0,23,1002.0
4,673d6028f83d1741dc6af737,0.0,23.0,0.0,run_1,182.0,4.0,1.0,1.0,3.0,...,2.0,2.0,1.0,0.2753,700.3703,2.0,0.2727,122.0,14,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41380,67038a1c7497dbfeaa1581cd,68.0,39.0,1.0,run_2,30.0,295.0,2.0,4.0,3.0,...,1.0,1.0,0.0,0.0223,2677.0169,2.0,0.0127,1211.0,32,1001.0
41381,67038a1c7497dbfeaa1581cd,68.0,39.0,1.0,run_2,60.0,296.0,2.0,2.0,3.0,...,1.0,1.0,1.0,0.0017,2678.7936,1.0,0.1305,1130.0,23,1000.0
41382,67038a1c7497dbfeaa1581cd,68.0,39.0,1.0,run_2,195.0,297.0,3.0,2.0,3.0,...,2.0,2.0,0.0,0.1777,2680.6789,1.0,0.1461,212.0,23,2.0
41383,67038a1c7497dbfeaa1581cd,68.0,39.0,1.0,run_2,84.0,298.0,2.0,2.0,3.0,...,1.0,1.0,0.0,0.1474,2682.7503,2.0,0.0221,1220.0,32,1000.0
